<a href="https://colab.research.google.com/github/JuanZapa7a/AINavalEngineering/blob/main/NB13_Convolutional_Neural_Networks_Underwater_Hull_Images.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **NB13 · Class 13 — Convolutional Neural Networks on Real Underwater Hull Images**

## Block 3: AI — Deep Learning (continued)

`NB11`/`NB12` trained networks on tabular data — the same kind of hand-engineered numeric features classical ML uses. This class is where Deep Learning earns its keep: **raw images**, where hand-engineering "the right features" would be far harder than it was for sonar frequency bands or hull-geometry coefficients. We use the real **[LIACi dataset](https://liaci.sintef.cloud)** (SINTEF Ocean, Norway): 1,893 real underwater ship-hull inspection images, pixel-level annotated across 11 categories (marine growth, corrosion, anodes, propellers, and more) — genuinely naval, genuinely unstructured data.

**Today's task**: a **binary image classifier** — does a given underwater image show a specific condition (we'll pick one class from the dataset, e.g. marine growth) or not? This is a deliberately simpler task than full pixel-level segmentation, scoped to fit a first hands-on CNN class; the full segmentation problem is real, harder, and left for later.

### Learning objectives

By the end of this class, students will be able to:
- Explain why a plain MLP is a poor fit for raw image data, and what a convolution operation actually computes.
- Explain the role of pooling layers in a CNN, and how stacking conv+pool layers builds up from pixels to high-level features.
- Load and prepare real image data for a PyTorch CNN (resizing, normalizing, building labels from the data itself).
- Build, train, and evaluate a small CNN classifier end to end, using the same rigor (train/val/test, dropout, honest evaluation) as `NB11`/`NB12`.
- Visualize what a trained CNN's early layers actually detect in a real image.

### Agenda (2-hour class)

| # | Class segment | Approx. duration | Type |
|---|---------------------|:---:|:---:|
| 1 | Recap, today's roadmap | 5 min | Theory |
| 2 | Why CNNs for images? The limits of a plain MLP | 10 min | Theory |
| 3 | The convolution operation: kernels and feature maps | 15 min | Theory + Practice |
| 4 | Pooling and CNN architecture, end to end | 10 min | Theory |
| 5 | Loading and exploring the real dataset | 15 min | Practice |
| 6 | Building classification labels from the data itself | 10 min | Practice |
| 7 | Preparing image tensors and a train/val/test split | 10 min | Practice |
| 8 | Hands-on: building and training a CNN classifier | 25 min | Practice |
| 9 | Evaluating the CNN and visualizing what it learned | 15 min | Practice |
| 10 | Summary, homework, next class | 5 min | Theory |

> Timings are approximate guidance, not a strict script — there are no scheduled breaks. If we cover everything with time to spare, class ends early; that can happen and is fine.

> **Before we start**: this class downloads a real ~1 GB dataset and trains on real images, both slower than `NB11`/`NB12`'s tabular data. If a GPU is available, switch to it now (`Runtime → Change runtime type → T4 GPU`) — training will be noticeably faster.

---

## 1. Recap: where we are

- **`NB11`**: perceptron → MLP theory, a first real PyTorch classifier on tabular sonar data.
- **`NB12`**: training deep networks properly — validation monitoring, dropout, early stopping, optimizers.
- **`NB13`** (today): the architecture built specifically for images — Convolutional Neural Networks — on real underwater inspection photos.

---

## 2. Why CNNs for images? The limits of a plain MLP

Nothing stops us from feeding a raw image into the `SonarMLP`-style network from `NB11`: flatten a 96×96 RGB image into a vector of 96×96×3 = 27,648 numbers, and feed it to `nn.Linear`. Two real problems with that:

- **Parameter explosion**: a first hidden layer of just 64 units on that input would need over 1.7 million weights — for one layer, on small images. Real photos are far larger.
- **No notion of *where***: a flattened vector treats pixel (0,0) and pixel (50,50) as unrelated numbers. It has no built-in concept that nearby pixels are related, or that a pattern (an edge, a texture) means the same thing whether it appears in the top-left or bottom-right of the image.

A **Convolutional Neural Network (CNN)** fixes both: it slides small, shared filters across the image (far fewer parameters, since the same filter is reused everywhere), and by construction, it looks at *local neighborhoods* of pixels — building up from edges, to textures, to whole objects, entirely automatically. This is precisely the "raw, unstructured data" case flagged back in `NB11` §2 as where deep learning's automatic feature learning genuinely pays off.

> **Further reading**: [Convolutional neural network (Wikipedia)](https://en.wikipedia.org/wiki/Convolutional_neural_network).

---

## 3. The convolution operation: kernels and feature maps

A **kernel** (or filter) is a small matrix — say 3×3 — of learnable weights. Convolution slides it across the image, and at every position multiplies the kernel elementwise with the pixels underneath it, sums the result, and writes that single number into an output **feature map**. Different kernels detect different things (edges, corners, textures); a CNN *learns* the kernel values during training, exactly like it learns the weights in `NB11`'s perceptron — a kernel is nothing more than a small, shared set of weights.

Let's compute one, by hand, on a tiny synthetic example — a vertical edge (left half dark, right half light) and a classic vertical-edge-detecting kernel:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

input_grid = np.array([
    [0, 0, 0, 1, 1, 1],
    [0, 0, 0, 1, 1, 1],
    [0, 0, 0, 1, 1, 1],
    [0, 0, 0, 1, 1, 1],
    [0, 0, 0, 1, 1, 1],
    [0, 0, 0, 1, 1, 1],
])
kernel = np.array([
    [1, 0, -1],
    [1, 0, -1],
    [1, 0, -1],
])

def convolve2d_valid(img, k):
    kh, kw = k.shape
    h, w = img.shape
    out = np.zeros((h - kh + 1, w - kw + 1))
    for i in range(out.shape[0]):
        for j in range(out.shape[1]):
            out[i, j] = np.sum(img[i:i + kh, j:j + kw] * k)
    return out

output_grid = convolve2d_valid(input_grid, kernel)

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))

axes[0].imshow(input_grid, cmap="gray", vmin=0, vmax=1)
axes[0].set_title("Input (6x6) + one 3x3 kernel window")
for (i, j), val in np.ndenumerate(input_grid):
    axes[0].text(j, i, int(val), ha="center", va="center", color="red", fontsize=9)
axes[0].add_patch(patches.Rectangle((1.5, -0.5), 3, 3, linewidth=2, edgecolor="lime", facecolor="none"))
axes[0].set_xticks([]); axes[0].set_yticks([])

axes[1].imshow(output_grid, cmap="viridis")
axes[1].set_title("Output feature map (4x4)")
for (i, j), val in np.ndenumerate(output_grid):
    axes[1].text(j, i, int(val), ha="center", va="center", color="white", fontsize=9)
axes[1].add_patch(patches.Rectangle((1.5, -0.5), 1, 1, linewidth=2, edgecolor="lime", facecolor="none"))
axes[1].set_xticks([]); axes[1].set_yticks([])

plt.tight_layout()
plt.show()

The highlighted 3×3 window in the input produces exactly the highlighted single value in the output feature map — that's one position of the slide. Notice the output is large wherever the kernel window straddles the dark/light boundary (an edge!) and near zero in flat regions — this specific kernel is, quite literally, an edge detector. In a real CNN, we never design kernels by hand like this; **the network learns whatever kernel values minimize the loss**, which might detect edges, colors, textures, or nothing humanly interpretable at all.

> **Further reading**: [Kernel — image processing (Wikipedia)](https://en.wikipedia.org/wiki/Kernel_%28image_processing%29) · [`torch.nn.Conv2d` documentation](https://pytorch.org/docs/stable/generated/torch.nn.Conv2d.html).

---

## 4. Pooling and CNN architecture, end to end

A **pooling layer** shrinks a feature map by summarizing small regions — **max pooling** (the common default) keeps only the largest value in each small window (e.g., 2×2), halving both dimensions. Two benefits: it makes the network more tolerant of a feature shifting by a pixel or two (a small translation doesn't change which value was the max, usually), and it keeps the number of values manageable as we stack more layers.

A typical CNN repeats **Convolution → activation (ReLU) → Pooling** several times, each block working on a coarser, more abstract version of the image, then **flattens** the final feature maps into a vector and feeds them to a small MLP "head" (exactly the fully-connected layers from `NB11`) that makes the final prediction:

| Stage | What it does | Analogy from `NB11`/`NB12` |
|---|---|---|
| Conv layers | Learn local patterns (edges → textures → parts) | The "features" a classical model would need hand-engineered |
| ReLU | Non-linearity between layers | Same role as in the MLP |
| Pooling | Shrink and summarize | — new to CNNs |
| Flatten + Linear layers | Combine learned features into a final prediction | Identical to `NB11`'s `SonarMLP` head |

Everything from `NB12` (dropout, early stopping, Adam) still applies unchanged once we reach the fully-connected head — a CNN is a feature extractor bolted onto the same kind of network we already know how to train properly.

---

## 5. Loading and exploring the real dataset

The **LIACi dataset** is published directly by SINTEF Ocean for research use. The download is roughly 1 GB — this cell may take a few minutes:

In [ ]:
!wget -q -O liaci_data.zip https://liaci.sintef.cloud/download_data/data.zip
!unzip -oq liaci_data.zip
!ls

Underwater hull inspection is normally done one of two ways: sending a diver down to visually inspect the hull, or deploying a remotely operated vehicle (ROV) with a camera. Both are expensive, weather-dependent, and — for diver inspections — genuinely risky; as a result, real ships get inspected far less often than would be ideal, and a lot rides on a human expert correctly identifying every relevant condition in whatever footage is captured. This is precisely the kind of task — visual pattern recognition on real, messy, unstructured images, at a scale and consistency no small inspection team could match by hand — that motivated Deep Learning's entrance into this course back in `NB11` §2. A model that reliably flags "marine growth visible here" from a photo doesn't replace the inspector, but it can triage thousands of frames from a single ROV survey down to the handful that actually need a trained eye.

The archive extracts an `images/` folder (the raw photos) and a `masks/` folder (one subfolder per annotated class, containing a same-named mask file per image where that class is visible). Let's look at one real image:

In [ ]:
import os
from PIL import Image
import matplotlib.pyplot as plt

base_dir = "LIACi_dataset_pretty"
imgs_path = os.path.join(base_dir, "images")
masks_path = os.path.join(base_dir, "masks")

image_files = sorted(os.listdir(imgs_path))
print("Total images:", len(image_files))

sample_img = Image.open(os.path.join(imgs_path, image_files[0]))
plt.imshow(sample_img)
plt.title(f"Example: {image_files[0]}")
plt.axis("off")
plt.show()

And the annotated condition classes available as masks (excluding two auxiliary folders — `saliency` and `segmentation` — that hold derived data rather than a single semantic class):

In [ ]:
mask_classes = sorted(f for f in os.listdir(masks_path) if f not in {"saliency", "segmentation"})
print(mask_classes)

---

## 6. Building classification labels from the data itself

There is no ready-made "yes/no" label file — but we can build one from the masks: for a chosen class, an image gets label **1** if its mask for that class contains any non-zero pixel (the condition is visible somewhere in the photo), and **0** otherwise.

We'll target **marine growth** ([biofouling](https://en.wikipedia.org/wiki/Biofouling)) specifically, and it's worth pausing on why this condition, out of the 11 available, is a genuinely good choice for a first real classifier — not just visually distinctive, but operationally important: biofouling on a hull increases drag enough to cut a ship's speed by up to 10%, which can require up to a **40% increase in fuel** to compensate, and the US Navy alone estimates it costs around **$1 billion per year** in extra fuel, maintenance, and control measures. If that number sounds familiar, it should — it's the same physical effect behind the real `fuel_consumption`/`engine_efficiency` columns we modeled in `NB07` and clustered in `NB09`, just observed here from a camera instead of a fuel log. A model that reliably spots growth early is, in a very direct sense, a tool for keeping the numbers in those earlier notebooks lower.

We'll pick whichever class name contains "growth" in the archive, without hardcoding the exact spelling used:

In [ ]:
target_class = next(c for c in mask_classes if "growth" in c.lower())
print("Target class for our binary classifier:", target_class)

Compute the label for every image:

In [ ]:
import numpy as np

def label_for(fname, cls):
    # Masks are stored as .bmp, regardless of the source image's extension
    # (here, .jpg) -- matching on the exact filename would silently fail
    # every lookup, so we match on the filename stem instead.
    stem = os.path.splitext(fname)[0]
    mask_file = os.path.join(masks_path, cls, stem + ".bmp")
    if not os.path.exists(mask_file):
        return 0
    mask = np.array(Image.open(mask_file).convert("L"))
    return int(mask.max() > 0)

all_labels = {f: label_for(f, target_class) for f in image_files}
positive = sum(all_labels.values())
print(f"{positive} / {len(all_labels)} images show '{target_class}'")

**Read your own output**: is the split roughly balanced, or skewed toward one class? A real dataset built this way often *isn't* perfectly balanced — biofouling severity varies a lot by hull location, time since last cleaning, and vessel history, so it would be unrealistic to expect a clean 50/50 split. This matters for two concrete reasons, not just an abstract warning: first, `NB07`'s point that accuracy alone is a poor metric for imbalanced problems still applies (a classifier that always predicts "Absent" could still score a high accuracy if the condition is rare in the data); second, when we subsample down to a manageable training set in Part 7, a small *purely random* subsample of an already-imbalanced dataset can end up with barely any examples of the minority class at all — worth designing around deliberately, not just hoping for the best.

---

## 7. Preparing image tensors and a train/val/test split

Working with all 1,893 full-resolution images would be slow for a live class, so we take a subsample, resize every image to a small, consistent size, and normalize pixel values to 0–1. Given Part 6's warning, we subsample **deliberately, not purely randomly**: pulling separately from the positive and negative image lists guarantees both classes are meaningfully represented in our training data, regardless of how skewed the full dataset happens to be. This is a legitimate, common technique — distinct from data leakage, since we're only choosing *which real, unmodified images* to include, not fabricating or duplicating any of them:

In [ ]:
import random

random.seed(42)

positive_files = [f for f in image_files if all_labels[f] == 1]
negative_files = [f for f in image_files if all_labels[f] == 0]
print(f"Available in the full dataset: {len(positive_files)} positive, {len(negative_files)} negative")

n_per_class = min(len(positive_files), len(negative_files), 200)
sample_files = random.sample(positive_files, n_per_class) + random.sample(negative_files, n_per_class)
random.shuffle(sample_files)

IMG_SIZE = 96

def load_image(fname, size=IMG_SIZE):
    img = Image.open(os.path.join(imgs_path, fname)).convert("RGB").resize((size, size))
    return np.array(img, dtype=np.float32) / 255.0

images = np.stack([load_image(f) for f in sample_files])
labels = np.array([all_labels[f] for f in sample_files])

print(images.shape, labels.shape)
print("Class balance in our sample:", np.bincount(labels))

Convert to PyTorch tensors — note the dimension reorder: PyTorch expects images as `(batch, channels, height, width)`, while NumPy/PIL give us `(batch, height, width, channels)`:

In [ ]:
import torch

X_img = torch.tensor(images).permute(0, 3, 1, 2)
y_img = torch.tensor(labels, dtype=torch.float32).view(-1, 1)
X_img.shape

Split 60/20/20, exactly like `NB12`, stratifying so both classes stay represented in every split:

In [ ]:
from sklearn.model_selection import train_test_split

idx = np.arange(len(labels))
idx_train_full, idx_test = train_test_split(idx, test_size=0.2, random_state=42, stratify=labels)
idx_train, idx_val = train_test_split(
    idx_train_full, test_size=0.25, random_state=42, stratify=labels[idx_train_full]
)

X_train, y_train = X_img[idx_train], y_img[idx_train]
X_val, y_val = X_img[idx_val], y_img[idx_val]
X_test, y_test = X_img[idx_test], y_img[idx_test]

X_train.shape, X_val.shape, X_test.shape

---

## 8. Hands-on: building and training a CNN classifier

Three Conv+ReLU+Pool blocks (16 → 32 → 64 filters, growing deeper as the spatial size shrinks — a very common pattern), then a dropout-regularized fully-connected head, following `NB12`'s practice directly:

In [ ]:
import torch.nn as nn

class HullCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # 96 -> 48
            nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # 48 -> 24
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # 24 -> 12
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(64 * 12 * 12, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

torch.manual_seed(42)
cnn = HullCNN()
sum(p.numel() for p in cnn.parameters())

That parameter count is worth comparing to §2's "1.7 million weights for one MLP layer" estimate — convolution's shared filters keep a much deeper network far smaller, even with three convolutional layers plus a fully-connected head.

A brief note on why this particular shape (16 → 32 → 64 filters, three pooling steps): it follows a very common, empirically-motivated pattern in real CNN architectures — as pooling shrinks the spatial size (96 → 48 → 24 → 12), we *increase* the number of filters, so each layer keeps roughly the same "budget" of information even as it covers a coarser view of the image. Early layers, working on the full-resolution image, tend to need only a few filters to capture simple local patterns (edges, color transitions); deeper layers, working on a coarser but more semantically rich representation, benefit from more filters to represent a wider variety of learned shapes and textures. We'll see a hint of this directly in Part 9, when we visualize what the very first layer actually detects.

Train with the same recipe as `NB12`: Adam, binary cross-entropy, tracking train and validation loss every epoch:

In [ ]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(cnn.parameters(), lr=0.001)

n_epochs = 25
train_losses, val_losses = [], []

for epoch in range(n_epochs):
    cnn.train()
    optimizer.zero_grad()
    outputs = cnn(X_train)
    loss = criterion(outputs, y_train)
    loss.backward()
    optimizer.step()
    train_losses.append(loss.item())

    cnn.eval()
    with torch.no_grad():
        val_loss = criterion(cnn(X_val), y_val)
    val_losses.append(val_loss.item())

    print(f"Epoch {epoch + 1}/{n_epochs} - train loss: {loss.item():.4f} - val loss: {val_loss.item():.4f}")

Plot both curves, exactly as in `NB12`:

In [ ]:
plt.plot(train_losses, label="Training loss")
plt.plot(val_losses, label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss (binary cross-entropy)")
plt.title(f"CNN training — detecting '{target_class}'")
plt.legend()
plt.show()

---

## 9. Evaluating the CNN and visualizing what it learned

Evaluate on the untouched test set with the same tools as every classifier since `NB08` — and, since this is a real inspection task, it's worth reading the confusion matrix with the actual operational stakes in mind, not just as four abstract numbers:

- A **false negative** here (predicting "Absent" when growth is actually present) means a fouled hull section goes unflagged — the condition keeps costing fuel and drag until the next inspection catches it, or doesn't.
- A **false positive** (predicting "Present" when the hull is actually clean) means an inspector spends time double-checking a section that didn't need it — wasted effort, but a far cheaper mistake than the false negative above.

This asymmetry is exactly the kind of reasoning `NB07` introduced with the oil-spill example — in a real deployment you would likely tune the classification threshold (currently a flat `0.5` in the code below) to trade some false positives for fewer false negatives, rather than treating both error types as equally costly:

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

cnn.eval()
with torch.no_grad():
    test_preds = (torch.sigmoid(cnn(X_test)) > 0.5).float()

y_pred_np = test_preds.numpy().ravel()
y_test_np = y_test.numpy().ravel()

# labels=[0, 1] keeps both classes in the report even if one happens to be
# briefly absent from this particular split or from the predictions -- with a
# small sample this is a real possibility, not just defensive styling.
print(confusion_matrix(y_test_np, y_pred_np, labels=[0, 1]))
print()
print(classification_report(
    y_test_np, y_pred_np, labels=[0, 1], target_names=["Absent", "Present"], zero_division=0
))

Metrics summarize; they don't show you *which* images are being confused, or whether the mistakes look reasonable to a human eye. Look at a handful of real predictions side by side with the actual images — for a vision model, this kind of qualitative check matters as much as any single metric, and is standard practice before trusting a model with real inspection footage:

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for ax, i in zip(axes.ravel(), range(8)):
    img_show = X_test[i].permute(1, 2, 0).numpy()
    ax.imshow(img_show)
    ax.set_title(f"True: {int(y_test_np[i])}  Pred: {int(y_pred_np[i])}")
    ax.axis("off")
plt.tight_layout()
plt.show()

Finally, look *inside* the network: what does the very first convolutional layer respond to on a real test image? This is exactly the "automatically learned features" idea from §2, made visible — and a standard technique in real computer-vision practice for sanity-checking that a model is looking at something sensible, rather than exploiting some unrelated artifact of the images (a known failure mode: a model that "cheats" by picking up on, say, a camera watermark or a lighting difference instead of the actual condition being classified):

In [ ]:
sample_image = X_test[0].unsqueeze(0)
with torch.no_grad():
    first_layer_output = cnn.features[0](sample_image)  # after the first Conv2d, before ReLU/pool

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for ax, ch in zip(axes.ravel(), range(8)):
    ax.imshow(first_layer_output[0, ch].numpy(), cmap="viridis")
    ax.set_title(f"Filter {ch}")
    ax.axis("off")
plt.suptitle("What the first conv layer's 8 filters respond to, on one real image")
plt.tight_layout()
plt.show()

**Read your own output**: some filters likely highlight edges or boundaries in the image, others might respond to color/texture regions, and some may look nearly blank on this particular image — normal for an untuned network trained briefly on a small sample. No one *designed* these filters; §3's edge detector was hand-built to make a point, but every filter here was learned purely by gradient descent minimizing the classification loss.

This pattern — early layers detecting simple, generic features (edges, colors, textures) that get combined into increasingly complex, task-specific patterns in deeper layers — was documented rigorously in real trained networks (not toy examples) by [Zeiler & Fergus (2014), *Visualizing and Understanding Convolutional Networks*](https://arxiv.org/abs/1311.2901), one of the papers that made CNN interpretability a serious research topic rather than a curiosity. It's also the reason **transfer learning** works at all: a later class in this block will reuse a network pretrained on millions of unrelated photos, on the reasonable assumption that its early, generic edge/texture filters are useful for almost any image task, including ours.

---

## Class summary

- A plain MLP on raw pixels is parameter-heavy and ignores spatial structure; convolution fixes both by sliding small, shared, learned filters across the image.
- A kernel/filter produces a feature map; different filters detect different patterns, and the network learns the filter values, not us.
- Pooling shrinks feature maps and adds tolerance to small shifts; stacking Conv+ReLU+Pool blocks builds from edges to increasingly abstract features.
- Once features are extracted, the same fully-connected head, dropout, and training discipline from `NB11`/`NB12` applies unchanged.
- We built classification labels directly from a real, pixel-annotated dataset (rather than assuming labels exist), trained a real CNN, and looked inside it to see what it actually learned.

## For the next class (NB14)

We close Block 3 with **sequence models** (RNN/LSTM) for time-dependent data — a different kind of structure than images, where *order* rather than spatial locality is what the architecture needs to respect.

## Homework / Practice Ideas

1. Change `target_class` to a different mask folder (e.g., corrosion or another class from §5's printed list) and retrain — does the CNN do noticeably better or worse on that condition? Why might that be (think about how visually distinctive the condition is, and how balanced the classes are)?
2. Increase `sample_files` from 400 to 800 (or the full dataset, if you have time and a GPU) — does more data close the gap between training and validation loss?
3. Add a fourth Conv+ReLU+Pool block (128 filters) — remember to recompute the `Linear` layer's input size to match the new, smaller feature map.
4. Try `IMG_SIZE = 64` instead of 96 — how much faster does training run, and how much (if any) does test accuracy suffer?
5. In your own words, explain why we used `stratify=labels` in the train/val/test splits here, tying it back to the class-balance check in Part 6.

> ***As always: a model's numbers matter, but so does actually looking at what it got right and wrong — Part 9's image grid is not optional homework, it's part of evaluating any real vision model.***
